# 08 - Train Models


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('online_shoppers_intention.csv')
df = df.drop_duplicates().copy()
df['Weekend'] = df['Weekend'].astype(int)
df['Revenue'] = df['Revenue'].astype(int)

df['Total_Pages'] = df['Administrative'] + df['Informational'] + df['ProductRelated']
df['Total_Duration'] = (
    df['Administrative_Duration'] +
    df['Informational_Duration'] +
    df['ProductRelated_Duration']
)

df = pd.get_dummies(
    df,
    columns=['Month', 'VisitorType'],
    drop_first=True,
    dtype=int
)

X = df.drop('Revenue', axis=1)
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import numpy as np

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

k_values = list(range(1, 22, 2))
knn_scores = [
    cross_val_score(
        KNeighborsClassifier(n_neighbors=k),
        X_train_s, y_train, cv=5, scoring='f1'
    ).mean()
    for k in k_values
]
best_k = k_values[np.argmax(knn_scores)]

depth_values = [3, 5, 10, 15, 20, None]
dt_scores = [
    cross_val_score(
        DecisionTreeClassifier(max_depth=d, random_state=42),
        X_train, y_train, cv=5, scoring='f1'
    ).mean()
    for d in depth_values
]
best_depth = depth_values[np.argmax(dt_scores)]


models = {
    'KNN': KNeighborsClassifier(n_neighbors=best_k),
    'SVM': SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=best_depth, class_weight='balanced', random_state=42)
}

models['KNN'].fit(X_train_s, y_train)
models['SVM'].fit(X_train_s, y_train)
models['Logistic Regression'].fit(X_train_s, y_train)
models['Decision Tree'].fit(X_train, y_train)

print('Best KNN k:', best_k)
print('Best Decision Tree max_depth:', best_depth)
print('All models trained successfully.')


/Users/dhruvvatsa/hadoop-docker/python/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Best KNN k: 3
Best Decision Tree max_depth: 5
All models trained successfully.
